In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("PostgresConnector") \
    .config("spark.jars","/usr/lib/spark/jars/postgresql-42.7.4.jar")\
    .getOrCreate()


In [ ]:
# Question 38: Salary increase percentage on the last raise.
# Concept: Aggregation over dates
employee_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.employee",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})

department_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.department",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})

department_employee_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.department_employee",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})
salary_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.salary",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})
# Solution:
window_spec = Window.partitionBy("employee_id").orderBy(col("from_date").desc())
ranked = salary_df.withColumn("rn", row_number().over(window_spec))

cur = ranked.filter(col("rn") == 1).withColumnRenamed("amount", "cur_amount")
prev = ranked.filter(col("rn") == 2).withColumnRenamed("amount", "prev_amount")

cur.join(prev, "employee_id") \
    .withColumn("pct", ((col("cur_amount") - col("prev_amount")) / col("prev_amount")) * 100) \
    .show()
